In [91]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import os
import sys
sys.path.append('../../config')
from ceride_palettes import PALETTES

trials = pd.read_csv('../../data/norming_study/processed/b_pilot/trials.csv')

RESULTS_DIR = '../../results/norming_study/b_pilot'
os.makedirs(RESULTS_DIR, exist_ok=True)

In [92]:
print(f"number of participants: {trials['subjectID'].nunique()}")
print(f"number of unique scenarios: {trials['actionPhrase'].nunique()} of 160")

number of participants: 7
number of unique scenarios: 95 of 160


In [93]:
aw = trials[trials['sliderOrder'] == 'AW']
wa = trials[trials['sliderOrder'] == 'WA']

# per-item means from the correct condition + response counts
item_n = trials.groupby('itemID').size().rename('n')
item_means = (
    aw.groupby('itemID')['abilityResponse'].mean()
    .to_frame()
    .join(wa.groupby('itemID')['willingnessResponse'].mean())
    .join(trials.groupby('itemID')['actionPhrase'].first())
    .join(item_n)
    .reset_index()
    .dropna()
)

c0 = '#de6841'  # warm orange — AW
c1 = '#677aaa'  # blue-purple — WA
bg = '#e6dec7'
grid = '#d4ccb5'

import textwrap
def wrap(text, width=55):
    return '<br>'.join(textwrap.wrap(str(text), width))

trials['_q'] = trials['actionPhrase'].apply(lambda x: wrap(f'Can you {x}?'))
trials['_v'] = trials['vignette'].apply(wrap)
item_means['_q'] = item_means['actionPhrase'].apply(lambda x: wrap(f'Can you {x}?'))

pad = 4

fig = make_subplots(
    rows=2, cols=2,
    column_widths=[0.8, 0.2],
    row_heights=[0.2, 0.8],
    shared_xaxes=True,
    shared_yaxes=True,
    horizontal_spacing=0.02,
    vertical_spacing=0.02,
)

# individual responses — one trace per condition for legend
for sliderOrder, color, label in [('AW', c0, 'AW (ability first)'), ('WA', c1, 'WA (willingness first)')]:
    g = trials[trials['sliderOrder'] == sliderOrder]
    fig.add_trace(go.Scatter(
        x=g['abilityResponse'],
        y=g['willingnessResponse'],
        mode='markers',
        name=label,
        marker=dict(color=color, size=4, opacity=0.6, line=dict(width=0)),
        customdata=np.stack([g['_v'], g['_q']], axis=1),
        hovertemplate='<i>%{customdata[0]}</i><br><br><b>"%{customdata[1]}"</b><br>ability: %{x} &nbsp;|&nbsp; willingness: %{y}<extra></extra>',
        legendgroup=sliderOrder,
        showlegend=True,
    ), row=2, col=1)

# per-item means
fig.add_trace(go.Scatter(
    x=item_means['abilityResponse'],
    y=item_means['willingnessResponse'],
    mode='markers',
    name='item mean',
    marker=dict(color='#464548', size=7, opacity=0.8, line=dict(color='white', width=1)),
    customdata=np.stack([item_means['_q'], item_means['n'].astype(int)], axis=1),
    hovertemplate='<b>"%{customdata[0]}"</b><br>ability: %{x:.1f} &nbsp;|&nbsp; willingness: %{y:.1f}<br>n = %{customdata[1]}<extra></extra>',
    showlegend=False,
), row=2, col=1)

fig.add_trace(go.Scatter(
    x=[0, 100], y=[0, 100],
    mode='lines',
    line=dict(color='#464548', width=0.8, dash='dash'),
    opacity=0.4,
    showlegend=False,
    hoverinfo='skip',
), row=2, col=1)

# marginals — stacked by condition
for sliderOrder, color in [('AW', c0), ('WA', c1)]:
    g = trials[trials['sliderOrder'] == sliderOrder]
    fig.add_trace(go.Histogram(
        x=g['abilityResponse'],
        xbins=dict(start=0, end=100, size=5),
        marker_color=color, opacity=0.7,
        legendgroup=sliderOrder, showlegend=False,
    ), row=1, col=1)
    fig.add_trace(go.Histogram(
        y=g['willingnessResponse'],
        ybins=dict(start=0, end=100, size=5),
        marker_color=color, opacity=0.7,
        legendgroup=sliderOrder, showlegend=False,
    ), row=2, col=2)

axis_style = dict(showgrid=True, gridcolor=grid, gridwidth=1,
                  linecolor='#464548', linewidth=1, ticks='outside',
                  tickcolor='#464548')

fig.update_layout(
    barmode='stack',
    width=580, height=580,
    plot_bgcolor=bg,
    paper_bgcolor='white',
    margin=dict(l=60, r=20, t=20, b=90),
    font=dict(family='Avenir, Helvetica Neue, Arial'),
    legend=dict(
        orientation='h',
        x=0.5, y=-0.13,
        xanchor='center', yanchor='top',
        bgcolor='rgba(255,255,255,0.85)',
        bordercolor='#ccc', borderwidth=1,
        font=dict(size=12),
    ),
    hoverlabel=dict(
        bgcolor='rgba(255,255,255,0.45)',
        bordercolor='#464548',
        font=dict(family='Avenir, Helvetica Neue, Arial', size=13, color='#222'),
        namelength=0,
    ),
)
fig.update_xaxes(range=[-pad, 100+pad], title_text='ability response', **axis_style, row=2, col=1)
fig.update_yaxes(range=[-pad, 100+pad], title_text='willingness response', **axis_style, row=2, col=1)
fig.update_xaxes(showticklabels=False, showgrid=False, row=1, col=1)
fig.update_yaxes(showticklabels=False, showgrid=False, row=2, col=2)

fig.write_html(os.path.join(RESULTS_DIR, 'scatter_ability_willingness.html'))
fig.write_image(os.path.join(RESULTS_DIR, 'scatter_ability_willingness.png'), scale=2)
fig.show()

In [94]:
print('P(able):',           aw['abilityResponse'].mean())
print('P(willing):',        wa['willingnessResponse'].mean())
print('P(willing | able):', aw['willingnessResponse'].mean())
print('P(able | willing):', wa['abilityResponse'].mean())
print('P(able ∩ willing):', (trials['abilityResponse'] * trials['willingnessResponse'] / 100).mean())
print('r(able, willing):',  item_means[['abilityResponse', 'willingnessResponse']].corr().iloc[0, 1])

P(able): 81.37777777777778
P(willing): 39.483333333333334
P(willing | able): 53.888888888888886
P(able | willing): 62.733333333333334
P(able ∩ willing): 35.91142857142857
r(able, willing): 0.05955060350172586
